# PYNQ-Z2 HLS m_axi 简单测试

这个 Notebook 只有两个主要代码块：

1. 载入 `base_add.bit` 和 HLS IP
2. 运行 IP、读取 buffer、检查数据、画图、计算基础幅值

请把 `base_add.bit`、`base_add.hwh` 和本 Notebook 放在同一个目录。

In [ ]:
# 代码块 1：载入 bit/hwh 和 IP

from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
import time

BIT_FILE = "base_add.bit"
IP_NAME = "base_add_0"

overlay = Overlay(BIT_FILE)
ip = getattr(overlay, IP_NAME)

print("Overlay loaded:", BIT_FILE)
print("IP list:", list(overlay.ip_dict.keys()))
print(ip.register_map)


In [ ]:
# 代码块 2：运行 IP、读取 buffer、检查数据、画图、计算基础结果

MAX_SAMPLE_N = 1024
BUFFER_WORDS = MAX_SAMPLE_N * 2
SAMPLE_COUNT = 1024

# HLS register addresses
AP_CTRL_ADDR = 0x00
BUFFER_ADDR = 0x10
SAMPLE_COUNT_ADDR = 0x18

AP_START = 0x01
AP_DONE = 0x02

sample_count = min(SAMPLE_COUNT, MAX_SAMPLE_N)

# Allocate DDR buffer for two channels
buf = allocate(shape=(BUFFER_WORDS,), dtype=np.int32)
buf[:] = -12345
buf.flush()

phys_addr = int(buf.physical_address)
print("Buffer physical address:", hex(phys_addr))

# Write arguments to HLS IP
ip.write(BUFFER_ADDR, phys_addr & 0xFFFFFFFF)
ip.write(SAMPLE_COUNT_ADDR, sample_count)

# Start HLS IP
print("CTRL before:", hex(ip.read(AP_CTRL_ADDR)))
ip.write(AP_CTRL_ADDR, AP_START)

# Wait for ap_done
start = time.time()
while (ip.read(AP_CTRL_ADDR) & AP_DONE) == 0:
    if time.time() - start > 2.0:
        buf.freebuffer()
        raise TimeoutError("HLS IP timeout waiting for ap_done")

print("CTRL after:", hex(ip.read(AP_CTRL_ADDR)))

# Read back DDR buffer
buf.invalidate()
ch0 = np.array(buf[:sample_count], dtype=np.int32)
ch1 = np.array(buf[MAX_SAMPLE_N:MAX_SAMPLE_N + sample_count], dtype=np.int32)

print("CH0 first 8:", ch0[:8])
print("CH1 first 8:", ch1[:8])

# Basic analysis
def basic_stats(x, name):
    x = np.asarray(x, dtype=np.float64)
    mean = np.mean(x)
    vpp = np.max(x) - np.min(x)
    rms_ac = np.sqrt(np.mean((x - mean) ** 2))
    print(f"{name}: mean={mean:.3f}, vpp={vpp:.3f}, rms_ac={rms_ac:.3f}")
    return mean, vpp, rms_ac

mean0, vpp0, rms0 = basic_stats(ch0, "CH0")
mean1, vpp1, rms1 = basic_stats(ch1, "CH1")

if vpp0 != 0:
    print("Vpp gain CH1/CH0:", vpp1 / vpp0)

# Plot only first 512 points for readability
plot_n = min(512, sample_count)
plt.figure(figsize=(10, 4))
plt.plot(ch0[:plot_n], label="CH0")
plt.plot(ch1[:plot_n], label="CH1")
plt.title("HLS m_axi Fake Capture")
plt.xlabel("Sample Index")
plt.ylabel("Sample Value")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Optional correctness check for the ramp/reverse-ramp HLS version.
# If you changed HLS to triangle wave, this check will naturally fail, so it only prints a warning.
ramp_ok = np.all(ch0 == np.arange(sample_count, dtype=np.int32)) and np.all(ch1 == np.arange(sample_count - 1, -1, -1, dtype=np.int32))
print("Ramp pattern check:", "PASS" if ramp_ok else "SKIP/FAIL, maybe HLS is not ramp version")

# Free buffer when done
buf.freebuffer()
